# 02 — Data Preprocessing, SOH & RUL Computation
This notebook loads the engineered cycle features, calculates State of Health (SOH %) and Remaining Useful Life (RUL), and demonstrates strict anti-leakage scaling.


In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../src'))
from preprocessing import compute_soh, compute_rul, AntiLeakageScaler, get_lobo_splits

data_path = os.path.abspath('../data/processed/cycle_features.csv')
df = pd.read_csv(data_path)
df = compute_soh(df)
df = compute_rul(df, eol_threshold_ah=1.4)
df.head()


### Inspecting SOH (%) Degradation Curves


In [ ]:
plt.figure(figsize=(10, 5))
for bat_id, grp in df.groupby('battery_id'):
    plt.plot(grp['cycle_number'], grp['soh_percent'], label=f"{bat_id}")

plt.axhline(y=70.0, color='gray', linestyle=':', label='70% SOH EOL')
plt.title("Battery State of Health (SOH %) vs Discharge Cycles", fontsize=12)
plt.xlabel("Discharge Cycle", fontsize=11)
plt.ylabel("SOH (%)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()


### Demonstrating Anti-Leakage Feature Scaling


In [ ]:
# Verify that scaler fits ONLY on train batteries
train_df, test_df, test_battery = next(get_lobo_splits(df))
print(f"Test Battery: {test_battery}")
print(f"Train samples: {len(train_df)} | Test samples: {len(test_df)}")

feature_cols = ['voltage_mean', 'discharge_duration', 'temp_mean']
scaler = AntiLeakageScaler(scaler_type='minmax')
X_train_scaled = scaler.fit_transform(train_df[feature_cols].values)
X_test_scaled = scaler.transform(test_df[feature_cols].values)

print(f"Train Scaled Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
print("Strict anti-leakage scaling confirmed.")
